# Importação Retroativa de Dados
Objetivo: Esse pipeline tem como objetivo realizar a importação retroativa de dados de vendas da Hotmart e TMB

Conversion types: 8

# 0. Importando Dependências

In [ ]:
# Importações básicas
import math
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

# Adiciona src ao path
sys.path.append('../src')


# Utilitários de dados
from data_utils import (
    load_raw_data,
    save_processed_data,
    remove_duplicates,
    handle_missing_values,
    detect_outliers,
    normalize_column,
    process_phone_string,
    process_phone_number,
    clean_and_lower_column,
    flatten_list_to_df,
    remove_buyers_from_dataframe
)

CRONOGRAMA_SUBDOMAIN = 'cronogramadosfluentes-xwamel'

# Utilitários SQL
from sql_utils import DatabaseConnection as Dbc, load_query_from_file

# Utilitários de visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Utilitários de API
from api_utils import (
    make_request,
    get_json,
    post_json,
    paginated_request,
    response_to_dataframe
)

# utilitários hotmart
from hotmart_utils import Hotmart, read_hotmart_csv, format_hotmart_conversions

# utilitários tmb
from tmb_utils import TMB   

# utilitários tally
from tally_utils import Tally  

# Configurações pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Load Database Driver
db = Dbc()

# Inicializar API Hotmart
hotmart = Hotmart()
# Inicializar API TMB
tmb = TMB()

# Inicializar API Tally Forms
tally = Tally()

# 1. Recuperando Vendas da Hotmart

In [ ]:
# ## Constates do Código
CONVERSION_TYPE_ID = "8"

# # Configurações de Importação
DESTINATION_QUEUE = 'S.NORM.PROD.NEW'
SKIP_ORQUESTRATION = True
SKIP_CONVERSION = False

# ## Recuperando Submissões via CSV
df_hotmart = read_hotmart_csv("hotmart_sales_1.csv", sep=";")

# # Transactions
transactions = pd.DataFrame(db.execute_query(query="select converted_producer_comission_value, transaction, checkout_platform, campaign_id from views.vw_conversions_type_8;"))

# # ---- CORREÇÃO DO FILTRO DE TRANSAÇÕES JÁ IMPORTADAS ----
# # O erro "Unalignable boolean Series provided as indexer" acontece pois o filtro estava tentando indexar o dataframe 'transactions'
# # usando um boolean Series do mesmo tamanho de df_hotmart['Transação']. Para filtrar linhas de df_hotmart usando valores vindos de transactions, 
# # você deve filtrar df_hotmart e NÃO transactions. O correto é:
# #   removemos do df_hotmart as transações que JÁ EXISTEM em transactions.

# # Passo 1: Transações já importadas (existentes no banco)
already_imported_transactions = set(transactions['transaction'].unique())

# # Passo 2: Filtra do dataframe original df_hotmart apenas as não importadas ainda
df_hotmart = df_hotmart[~df_hotmart['Transação'].isin(already_imported_transactions)].copy()

df_hotmart_conversions = format_hotmart_conversions(df_hotmart)
df_hotmart_conversions = flatten_list_to_df(df_hotmart_conversions)

In [ ]:
import requests
import time
import json
import os

IMPOT_DATAFRAME = df_hotmart_conversions
BASE_URL = "https://southamerica-east1-aloud-etl.cloudfunctions.net/identity-resolution-http"

responses = []

PERSISTENCE_FILE = "imported_leads.json"

# Função auxiliar para obter valor de chave profunda (key_path)
def get_deep_key(d, key_path, default=""):
    """Busca valor em d pelo key_path em formato dot (ex: 'conversion_data.conversion_raw_info.transaction')"""
    try:
        for key in key_path.split("."):
            d = d[key]
        return d
    except (KeyError, TypeError):
        return default

# Permite selecionar qual chave (via path) será usada para controle de importação
IMPORT_CONTROL_KEY_PATH = "conversion_data.conversion_raw_info.transaction"

# Carregue os registros persistidos previamente já enviados, se existir
if os.path.exists(PERSISTENCE_FILE):
    with open(PERSISTENCE_FILE, "r", encoding="utf-8") as f:
        imported_keys = set(json.load(f))
else:
    imported_keys = set()

total_to_import = len(IMPOT_DATAFRAME)
print(f"Total de registros para importar: {total_to_import}")

for idx, item in enumerate(IMPOT_DATAFRAME, start=1):
    import_key = str(get_deep_key(item, IMPORT_CONTROL_KEY_PATH, "")).strip()
    if not import_key:
        print(f"Registro {idx} ignorado: chave de controle '{IMPORT_CONTROL_KEY_PATH}' ausente ou inválida.")
        continue
    if import_key in imported_keys:
        print(f"Registro {idx} já foi importado anteriormente para a chave '{import_key}', ignorando.")
        continue
    
    print(f"Importando registro {idx} de {total_to_import}... (key: {import_key})")
    response = requests.post(BASE_URL, json=item)
    try:
        resp_json = response.json()
    except Exception:
        resp_json = response.text
    responses.append(resp_json)
    
    # Atualize a persistência local imediatamente
    imported_keys.add(import_key)
    with open(PERSISTENCE_FILE, "w", encoding="utf-8") as f:
        json.dump(sorted(list(imported_keys)), f, ensure_ascii=False, indent=2)